**Модель Natasha/Slovnet NER**

In [ ]:
!pip install natasha razdel pandas

3 группы параметров:
1) Предобработка: ["raw", "lowercase", "clean"]

2) Разбиение текста: ["whole_text", "sentence_split"]

3) Batch size: [1, 8, 16]

Для каждого эксперимента считается
Precision
Recall
F1
micro-F1
macro-F1
per-class F1
время работы
texts/sec

Лучший для таблицы выбран по metrics["micro_f1"]

**Обычный тест**

In [ ]:
import ast
import re
import time
import pandas as pd
from razdel import sentenize
from natasha import Segmenter, NewsEmbedding, NewsNERTagger, Doc


ENTITY_CLASSES = ["PERSON", "PHONE", "EMAIL", "ADDRESS", "ID"]


def load_test_file(path: str) -> pd.DataFrame:

    try:
        df = pd.read_csv(path, sep="\t")
        if "text" not in df.columns:
            df = pd.read_csv(path)
    except Exception:
        df = pd.read_csv(path)

    df["label"] = df["label"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

    df = df.dropna(subset=["text"])

    df = df[df["text"].astype(str).str.strip() != ""]

    df = df.reset_index(drop=True)

    return df

In [ ]:
def preprocess_text(text: str, mode: str) -> str:
    if mode == "raw":
        return text

    if mode == "lowercase":
        return text.lower()

    if mode == "clean":
        text = re.sub(r"\s+", " ", text)
        text = text.strip()
        return text


In [ ]:
class NatashaSlovnetNER:
    def __init__(self):
        self.segmenter = Segmenter()
        self.emb = NewsEmbedding()
        self.ner_tagger = NewsNERTagger(self.emb)

    def predict_one(self, text: str, split_mode: str = "whole_text"):

        entities = []

        if split_mode == "whole_text":
            doc = Doc(text)
            doc.segment(self.segmenter)
            doc.tag_ner(self.ner_tagger)

            for span in doc.spans:
                if span.type == "PER":
                    entities.append({
                        "start": span.start,
                        "end": span.stop,
                        "label": "PERSON"
                    })

        elif split_mode == "sentence_split":
            for sent in sentenize(text):
                sent_text = sent.text
                offset = sent.start

                doc = Doc(sent_text)
                doc.segment(self.segmenter)
                doc.tag_ner(self.ner_tagger)

                for span in doc.spans:
                    if span.type == "PER":
                        entities.append({
                            "start": offset + span.start,
                            "end": offset + span.stop,
                            "label": "PERSON"
                        })

        else:
            raise ValueError(f"Unknown split_mode: {split_mode}")

        return entities

    def predict_batch(self, texts, batch_size: int = 8, split_mode: str = "whole_text"):
        all_predictions = []

        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            for text in batch:
                all_predictions.append(self.predict_one(text, split_mode=split_mode))

        return all_predictions

In [ ]:
def normalize_gold(label_list):

    if label_list is None:
        return []

    if isinstance(label_list, float) and pd.isna(label_list):
        return []

    if isinstance(label_list, str):
        label_list = ast.literal_eval(label_list)

    return [
        {
            "start": int(x[0]),
            "end": int(x[1]),
            "label": x[2]
        }
        for x in label_list
    ]


def entity_key(ent):
    return (ent["start"], ent["end"], ent["label"])

In [ ]:
def calculate_metrics(gold_all, pred_all, classes=ENTITY_CLASSES):
    per_class = {}

    total_tp = 0
    total_fp = 0
    total_fn = 0

    for cls in classes:
        tp = fp = fn = 0

        for gold, pred in zip(gold_all, pred_all):
            gold_cls = {entity_key(e) for e in gold if e["label"] == cls}
            pred_cls = {entity_key(e) for e in pred if e["label"] == cls}

            tp += len(gold_cls & pred_cls)
            fp += len(pred_cls - gold_cls)
            fn += len(gold_cls - pred_cls)

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if (precision + recall) > 0
            else 0.0
        )

        per_class[cls] = {
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
            "TP": tp,
            "FP": fp,
            "FN": fn,
        }

        total_tp += tp
        total_fp += fp
        total_fn += fn

    micro_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    micro_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    micro_f1 = (
        2 * micro_precision * micro_recall / (micro_precision + micro_recall)
        if (micro_precision + micro_recall) > 0
        else 0.0
    )

    macro_f1 = sum(per_class[c]["F1"] for c in classes) / len(classes)

    return {
        "per_class": per_class,
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
    }

In [ ]:
def run_experiment(
    df,
    preprocessing_mode: str = "raw",
    split_mode: str = "whole_text",
    batch_size: int = 8
):
    model = NatashaSlovnetNER()

    texts = [preprocess_text(t, preprocessing_mode) for t in df["text"].tolist()]
    gold = [normalize_gold(x) for x in df["label"].tolist()]



    start_time = time.time()
    predictions = model.predict_batch(
        texts,
        batch_size=batch_size,
        split_mode=split_mode
    )
    elapsed = time.time() - start_time

    metrics = calculate_metrics(gold, predictions)

    return metrics, predictions, elapsed


def metrics_to_table(metrics):
    rows = []

    for cls in ENTITY_CLASSES:
        cls_metrics = metrics["per_class"][cls]
        rows.append({
            "Класс": cls,
            "Precision": round(cls_metrics["Precision"], 4),
            "Recall": round(cls_metrics["Recall"], 4),
            "F1": round(cls_metrics["F1"], 4),
            "TP": cls_metrics["TP"],
            "FP": cls_metrics["FP"],
            "FN": cls_metrics["FN"],
        })

    return pd.DataFrame(rows)

In [ ]:
def build_person_errors_df(df: pd.DataFrame, predictions):
    rows = []

    gold_all = [normalize_gold(x) for x in df["label"].tolist()]

    for i, (text, gold, pred) in enumerate(zip(df["text"].astype(str), gold_all, predictions)):
        gold_person = [e for e in gold if e["label"] == "PERSON"]
        pred_person = [e for e in pred if e["label"] == "PERSON"]

        gold_set = {entity_key(e) for e in gold_person}
        pred_set = {entity_key(e) for e in pred_person}

        fp = [p for p in pred_person if entity_key(p) not in gold_set]
        fn = [g for g in gold_person if entity_key(g) not in pred_set]

        if fp or fn:
            rows.append({
                "id": i,
                "source": df.loc[i, "source"] if "source" in df.columns else None,
                "text": text,
                "false_positive": [
                    {**e, "text": text[e["start"]:e["end"]]}
                    for e in fp
                ],
                "false_negative": [
                    {**e, "text": text[e["start"]:e["end"]]}
                    for e in fn
                ],
            })

    return pd.DataFrame(rows)


def print_person_errors(errors_df: pd.DataFrame, limit=None):
    print(f"Количество текстов с ошибками: {len(errors_df)}")

    shown = errors_df if limit is None else errors_df.head(limit)

    for _, row in shown.iterrows():
        print(f"ID: {row['id']}")

        if row.get("source"):
            print(f"Source: {row['source']}")

        print(f"Текст: {row['text']}")

        if row["false_positive"]:
            print("FP:")
            for e in row["false_positive"]:
                print(f'  "{e["text"]}" [{e["start"]}:{e["end"]}] → PERSON')

        if row["false_negative"]:
            print("FN:")
            for e in row["false_negative"]:
                print(f'  "{e["text"]}" [{e["start"]}:{e["end"]}] → PERSON')

In [ ]:
def run_all_experiments(path: str):
    df = load_test_file(path)

    preprocessing_modes = ["raw", "lowercase", "clean"]
    split_modes = ["whole_text", "sentence_split"]
    batch_sizes = [1, 8, 16]

    summary_rows = []
    best_result = None

    for prep in preprocessing_modes:
        for split in split_modes:
            for batch_size in batch_sizes:
                metrics, predictions, elapsed = run_experiment(
                    df=df,
                    preprocessing_mode=prep,
                    split_mode=split,
                    batch_size=batch_size
                )

                row = {
                    "preprocessing": prep,
                    "split_mode": split,
                    "batch_size": batch_size,
                    "micro_precision": round(metrics["micro_precision"], 4),
                    "micro_recall": round(metrics["micro_recall"], 4),
                    "micro_f1": round(metrics["micro_f1"], 4),
                    "macro_f1": round(metrics["macro_f1"], 4),
                    "time_sec": round(elapsed, 4),
                    "texts_per_sec": round(len(df) / elapsed, 4) if elapsed > 0 else None,
                }

                summary_rows.append(row)

                if best_result is None or metrics["micro_f1"] > best_result["metrics"]["micro_f1"]:
                    best_result = {
                        "settings": row,
                        "metrics": metrics,
                        "predictions": predictions,
                        "time": elapsed,
                    }

    summary_df = pd.DataFrame(summary_rows)
    summary_df = summary_df.sort_values(
        by=["micro_f1", "macro_f1", "texts_per_sec"],
        ascending=False
    )

    print(summary_df.to_string(index=False))

    print(best_result["settings"])

    class_table = metrics_to_table(best_result["metrics"])
    print(class_table.to_string(index=False))


    person_errors_df = build_person_errors_df(
    df,
    best_result["predictions"]
    )

    print_person_errors(person_errors_df, limit=100)

    best_result["person_errors_df"] = person_errors_df

    return summary_df, class_table, best_result

In [ ]:
if __name__ == "__main__":
    summary_df, class_table, best_result = run_all_experiments("/content/test_small - test_final_300_v3.csv.csv")

    summary_df.to_csv("slovnet_experiments_summary.csv", index=False)
    class_table.to_csv("slovnet_best_per_class_metrics.csv", index=False)

    best_result["person_errors_df"].to_csv(
        "slovnet_person_errors.csv",
        index=False
    )

Выводы:

1) Логично, что приведение текста к нижнему регистру приводит к полной деградации качества распознавания (F1 = 0).

2)Очистка текста приводит к значительному снижению качества (micro-F1 ≈ 0.18), что указывает на чувствительность модели к изменению исходной структуры текста.

3) Изменение размера батча не оказывает влияния на качество распознавания.

4) Разбиение текста на предложения не оказывает  влияния на качество модели.

macro-F1 низкий, так как модель не распознаёт большинство классов сущностей, и их F1 равен 0.

Наилучший результат достигается без дополнительной предобработки текста (raw).

Главное:  Несмотря на высокое качество распознавания сущностей типа PERSON (F1 ≈ 0.93), модель не обнаруживает другие типы персональных данных (PHONE, EMAIL, ADDRESS, ID).

**Tricky**

In [ ]:
model = NatashaSlovnetNER()
print(dir(model))

In [ ]:
errors_raw = print_tricky_raw_errors("/content/tricky_139_final_labeled - tricky_139_final_labeled_converted.csv (6).csv")

In [ ]:
import pandas as pd
import ast

def load_tricky_file(path):
    df = pd.read_csv(path)

    df = df.dropna(subset=["text"])
    df = df[df["text"].astype(str).str.strip() != ""]

    if "label" in df.columns:
        df["label"] = df["label"].apply(parse_labels)

    df = df.reset_index(drop=True)
    return df


def parse_labels(x):
    if x is None:
        return []

    if isinstance(x, float) and pd.isna(x):
        return []

    if isinstance(x, str):
        x = x.strip()
        if x == "" or x.lower() == "nan":
            return []
        return ast.literal_eval(x)

    return x


def normalize_entity(ent, text=None):
    if isinstance(ent, dict):
        start = int(ent["start"])
        end = int(ent["end"])
        label = ent["label"]
        ent_text = ent.get("text", text[start:end] if text else "")

    elif isinstance(ent, (list, tuple)) and len(ent) >= 3:
        start = int(ent[0])
        end = int(ent[1])
        label = ent[2]
        ent_text = text[start:end] if text else ""


    return {
        "start": start,
        "end": end,
        "label": label,
        "text": ent_text
    }


def find_false_positives(gold, preds, text):
    gold_set = set()

    for g in gold:
        g_norm = normalize_entity(g, text)
        gold_set.add((g_norm["start"], g_norm["end"], g_norm["label"]))

    fps = []

    for p in preds:
        p_norm = normalize_entity(p, text)
        key = (p_norm["start"], p_norm["end"], p_norm["label"])

        if key not in gold_set:
            fps.append(p_norm)

    return fps


def print_tricky_false_positives(
    path="/content/tricky_139_final_labeled - tricky_139_final_labeled_converted.csv (6).csv"
):
    df = load_tricky_file(path)
    model = NatashaSlovnetNER()

    rows = []
    total_fp = 0

    for i, row in df.iterrows():
        text = row["text"]
        gold = row["label"]

        preds = model.predict_one(text)

        fps = find_false_positives(gold, preds, text)

        if fps:
            total_fp += len(fps)

            rows.append({
                "id": i,
                "text": text,
                "fp_count": len(fps),
                "false_positives": "; ".join(
                    f'{fp["text"]} [{fp["start"]}:{fp["end"]}, {fp["label"]}]'
                    for fp in fps
                ),
                "gold": gold,
                "predictions": [
                    normalize_entity(p, text)
                    for p in preds
                ]
            })

    fp_df = pd.DataFrame(rows)

    print(f"Текстов с ложными срабатываниями: {len(fp_df)}")
    print(f"Всего ложных срабатываний: {total_fp}")

    return fp_df

In [ ]:
fp_df = print_tricky_false_positives(
    "/content/tricky_139_final_labeled - tricky_139_final_labeled_converted.csv (6).csv"
)

fp_df

Все ложные срабатывания относятся к фамилиеподобным словам, которые в контексте не обозначают физическое лицо. Это показывает ограничение модели Slovnet/Natasha: она хорошо распознаёт именные формы, но не всегда различает персональные данные и неперсональные употребления фамилий в адресах, названиях объектов и коммерческих наименованиях.

**No-PLL**

In [ ]:
import time
import pandas as pd
from razdel import sentenize
from natasha import Segmenter, NewsEmbedding, NewsNERTagger, Doc


ENTITY_CLASSES = ["PERSON", "PHONE", "EMAIL", "ADDRESS", "ID"]


def load_no_pd_file(path: str) -> pd.DataFrame:
    try:
        df = pd.read_csv(path, sep="\t")
        if "text" not in df.columns:
            df = pd.read_csv(path)
    except Exception:
        df = pd.read_csv(path)

    df = df[["text"]].copy()
    df["label"] = [[] for _ in range(len(df))]
    return df


def preprocess_text(text: str, mode: str) -> str:
    if mode == "raw":
        return text
    if mode == "clean":
        return " ".join(text.split())
    if mode == "lowercase":
        return text.lower()
    raise ValueError(f"Unknown mode: {mode}")


class NatashaSlovnetNER:
    def __init__(self):
        self.segmenter = Segmenter()
        self.emb = NewsEmbedding()
        self.ner_tagger = NewsNERTagger(self.emb)

    def predict_one(self, text: str, split_mode: str = "whole_text"):
        entities = []

        if split_mode == "whole_text":
            doc = Doc(text)
            doc.segment(self.segmenter)
            doc.tag_ner(self.ner_tagger)

            for span in doc.spans:
                if span.type == "PER":
                    entities.append({
                        "start": span.start,
                        "end": span.stop,
                        "label": "PERSON",
                        "text": text[span.start:span.stop]
                    })

        elif split_mode == "sentence_split":
            for sent in sentenize(text):
                doc = Doc(sent.text)
                doc.segment(self.segmenter)
                doc.tag_ner(self.ner_tagger)

                for span in doc.spans:
                    start = sent.start + span.start
                    end = sent.start + span.stop

                    if span.type == "PER":
                        entities.append({
                            "start": start,
                            "end": end,
                            "label": "PERSON",
                            "text": text[start:end]
                        })
        else:
            raise ValueError(f"Unknown split_mode: {split_mode}")

        return entities

    def predict_batch(self, texts):
        return [self.predict_one(text) for text in texts]


def entity_key(ent):
    return (ent["start"], ent["end"], ent["label"])


def calculate_metrics(gold_all, pred_all, classes=ENTITY_CLASSES):
    per_class = {}
    total_tp = total_fp = total_fn = 0

    for cls in classes:
        tp = fp = fn = 0

        for gold, pred in zip(gold_all, pred_all):
            gold_cls = {entity_key(e) for e in gold if e["label"] == cls}
            pred_cls = {entity_key(e) for e in pred if e["label"] == cls}

            tp += len(gold_cls & pred_cls)
            fp += len(pred_cls - gold_cls)
            fn += len(gold_cls - pred_cls)

        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

        per_class[cls] = {
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
            "TP": tp,
            "FP": fp,
            "FN": fn,
        }

        total_tp += tp
        total_fp += fp
        total_fn += fn

    micro_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 0.0
    micro_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 0.0
    micro_f1 = (
        2 * micro_precision * micro_recall / (micro_precision + micro_recall)
        if (micro_precision + micro_recall)
        else 0.0
    )
    macro_f1 = sum(per_class[c]["F1"] for c in classes) / len(classes)

    return {
        "per_class": per_class,
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
    }


def metrics_to_table(metrics):
    rows = []
    for cls in ENTITY_CLASSES:
        m = metrics["per_class"][cls]
        rows.append({
            "Класс": cls,
            "Precision": round(m["Precision"], 4),
            "Recall": round(m["Recall"], 4),
            "F1": round(m["F1"], 4),
            "TP": m["TP"],
            "FP": m["FP"],
            "FN": m["FN"],
        })
    return pd.DataFrame(rows)


def run_experiment(df, model, preprocessing_mode: str):
    original_texts = df["text"].tolist()
    processed_texts = [preprocess_text(t, preprocessing_mode) for t in original_texts]
    gold = df["label"].tolist()

    start = time.time()
    predictions = [model.predict_one(text, split_mode="whole_text") for text in processed_texts]
    elapsed = time.time() - start

    metrics = calculate_metrics(gold, predictions)

    fp_texts = sum(1 for p in predictions if len(p) > 0)
    fp_entities = sum(len(p) for p in predictions)

    error_rows = []
    for i, (original_text, processed_text, preds) in enumerate(
        zip(original_texts, processed_texts, predictions)
    ):
        if preds:
            error_rows.append({
                "id": i,
                "preprocessing": preprocessing_mode,
                "text": original_text,
                "processed_text": processed_text,
                "predictions": preds,
                "predicted_entities": "; ".join(
                    f'{p["text"]} [{p["start"]}:{p["end"]}, {p["label"]}]'
                    for p in preds
                )
            })

    return metrics, predictions, elapsed, fp_texts, fp_entities, pd.DataFrame(error_rows)


def run_no_pd_experiments(path: str):
    df = load_no_pd_file(path)
    model = NatashaSlovnetNER()

    preprocessing_modes = ["raw", "clean", "lowercase"]

    summary_rows = []
    error_tables = {}

    for prep in preprocessing_modes:
        metrics, predictions, elapsed, fp_texts, fp_entities, errors_df = run_experiment(
            df=df,
            model=model,
            preprocessing_mode=prep
        )

        summary_rows.append({
            "preprocessing": prep,
            "split_mode": "whole_text",
            "micro_precision": round(metrics["micro_precision"], 4),
            "micro_recall": round(metrics["micro_recall"], 4),
            "micro_f1": round(metrics["micro_f1"], 4),
            "macro_f1": round(metrics["macro_f1"], 4),
            "fp_texts": fp_texts,
            "fp_entities": fp_entities,
            "fp_text_rate": round(fp_texts / len(df), 4),
            "time_sec": round(elapsed, 4),
            "texts_per_sec": round(len(df) / elapsed, 4) if elapsed > 0 else None,
        })

        error_tables[prep] = errors_df

    summary_df = pd.DataFrame(summary_rows)

    raw_metrics, _, _, _, _, _ = run_experiment(df, model, "raw")
    class_table = metrics_to_table(raw_metrics)

    print(summary_df.to_string(index=False))

    print(class_table.to_string(index=False))

    raw_errors = error_tables["raw"]

    if raw_errors.empty:
        print("Ошибок нет: модель не нашла PERSON в no-PD текстах.")
    else:
        for _, row in raw_errors.iterrows():
            print("\n---")
            print(f"ID: {row['id']}")
            print(f"Предсказания: {row['predicted_entities']}")
            print(f"Текст: {row['text']}")

    summary_df.to_csv("slovnet_no_pd_summary.csv", index=False)
    class_table.to_csv("slovnet_no_pd_per_class_metrics_raw.csv", index=False)
    raw_errors.to_csv("slovnet_no_pd_errors_raw.csv", index=False)

    return summary_df, class_table, raw_errors


if __name__ == "__main__":
    summary_df, class_table, raw_errors = run_no_pd_experiments("/content/no_pd_eval_clean.csv")

На выборке текстов, не содержащих персональных данных, модель Slovnet/Natasha не обнаружила ни одного ложного срабатывания (FP texts = 0 из 208).